## **E. LLM Based Chunking**

### What is it?
- LLM-based chunking uses a language model to intelligently determine
where to split text based on semantic understanding. The LLM
analyzes the document and decides optimal chunk boundaries,
potentially generating summaries or extracting key information.
- This is like having an intelligent editor who understands the content
and knows exactly where topics begin and end.

> Advantages:
- Most intelligent and context-aware splitting
- Can add contextual summaries to chunks
- Understands semantic boundaries better than rules
- Can adapt to different document types
- Improves retrieval quality significantly
- Can extract key information

> Disadvantages:
- Very expensive (LLM API calls for every chunk)
- Slowest processing time
- Requires API access and costs money
- Not deterministic (results may vary)
- Overkill for simple documents
- Latency issues for large documents
- Complex to implement and maintain

In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel
from langchain_core.documents import Document
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate

In [2]:
load_dotenv()

True

In [3]:
text= """Artificial intelligence is transforming technology and shaping the future.
Machine learning algorithms are becoming more sophisticated every day.
Deep learning models can now process vast amounts of data efficiently.
Neural networks are inspired by the human brain's structule.
The best pasta recipes include fresh ingredients and proper cooking techniques.
Italian cuisine emphasizes quality olive oil and regional cheeses.
Authentic carbonara uses guanciale, eggs, pecorino romano, and black pepper.
Cooking pasta al dente ensures the best texture and flavor.
Climate change is affecting ecosystems worldwide.
Rising temperatures are causing glaciers to melt at unprecedented rates.
Scientists warn that immediate action is needed to reduce carbon emissions.
Renewable energy sources offer hope for a sustainable future. """

In [4]:
## Pydantic Class for Structure Output
class Chunk(BaseModel):
    chunk_text: str
    summary: str

class Chunker(BaseModel):
    chunks: list[Chunk]

In [5]:
model= ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")

llm_chunker= model.with_structured_output(schema= Chunker)

In [6]:
prompt = ChatPromptTemplate(
    messages=[
        ( "system",
        """You are an expert Text Chunker that splits the given text and outputs them as a
        list of strings. You understand the natural topic boundaries of text and
        also do not change the existing text. You just split the text where ever applicable.
        Once you create the chunk, you also generate a 1-2 line summary of the chunk also"""),
        ( "human",
        "Split the given text into chunks\nText: {text}")
    ], 
    input_variables=["text"]
)

### **LangChain Expression Language (LCEL)**
##### model_chain = prompt | llm_chunker (The Pipe Operator) 
- In LangChain, the | (pipe) operator is used to connect components together, similar to how command-line piping works. It means: "take the output of the left side and pass it as the input to the right side."
- Here, it links your prompt (which formats the user text into system/human messages) to llm_chunker (your Gemini model with structured Pydantic output).
- When data flows through this chain:
The input variables are formatted into the prompt template.
The formatted prompt is immediately sent to Gemini.
Gemini processes it and returns the structured Pydantic object.

##### response = model_chain.invoke({"text": text})
invoke(...) is the standard method used to run a LangChain chain.
You pass it a dictionary {"text": text} which contains the actual string you want to chunk. 
The key "text" matches the {text} placeholder you defined in your human prompt template.
Under the hood, this single line:
- Replaces {text} in your prompt template with your actual paragraph of text.
- Submits the fully constructed prompt to the Gemini model.
- The model processes the instructions, splits the text into chunks, summarizes each chunk, and populates the Pydantic Chunker schema.
- Returns the result as a structured Pydantic object stored in response (which you can easily access via response.chunks).

In [7]:
# chunking through llm
model_chain = prompt | llm_chunker

response = model_chain.invoke({"text": text})

In [8]:
model_chain

ChatPromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are an expert Text Chunker that splits the given text and outputs them as a\n        list of strings. You understand the natural topic boundaries of text and\n        also do not change the existing text. You just split the text where ever applicable.\n        Once you create the chunk, you also generate a 1-2 line summary of the chunk also'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='Split the given text into chunks\nText: {text}'), additional_kwargs={})])
| _ChatModelBinding(bound=ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11', 'langchain-google-genai': '4.2.6'}}, output_version=None, profile={'name': 'Gemini 3.1 Flash L

In [9]:
chunks= response.chunks
response

Chunker(chunks=[Chunk(chunk_text="Artificial intelligence is transforming technology and shaping the future. Machine learning algorithms are becoming more sophisticated every day. Deep learning models can now process vast amounts of data efficiently. Neural networks are inspired by the human brain's structule.", summary='This section covers advancements in artificial intelligence, including machine learning and neural networks.'), Chunk(chunk_text='The best pasta recipes include fresh ingredients and proper cooking techniques. Italian cuisine emphasizes quality olive oil and regional cheeses. Authentic carbonara uses guanciale, eggs, pecorino romano, and black pepper. Cooking pasta al dente ensures the best texture and flavor.', summary='This section discusses the fundamentals of preparing high-quality Italian pasta dishes.'), Chunk(chunk_text='Climate change is affecting ecosystems worldwide. Rising temperatures are causing glaciers to melt at unprecedented rates. Scientists warn that

In [10]:
len(response.chunks)

3

In [11]:
response.chunks[0].chunk_text

"Artificial intelligence is transforming technology and shaping the future. Machine learning algorithms are becoming more sophisticated every day. Deep learning models can now process vast amounts of data efficiently. Neural networks are inspired by the human brain's structule."

In [12]:
for i in range(0, len(response.chunks)):
    print(f"Chunk {i+1}: {response.chunks[i].chunk_text}")
    print(f"Summary {i+1}: {response.chunks[i].summary}\n")

Chunk 1: Artificial intelligence is transforming technology and shaping the future. Machine learning algorithms are becoming more sophisticated every day. Deep learning models can now process vast amounts of data efficiently. Neural networks are inspired by the human brain's structule.
Summary 1: This section covers advancements in artificial intelligence, including machine learning and neural networks.

Chunk 2: The best pasta recipes include fresh ingredients and proper cooking techniques. Italian cuisine emphasizes quality olive oil and regional cheeses. Authentic carbonara uses guanciale, eggs, pecorino romano, and black pepper. Cooking pasta al dente ensures the best texture and flavor.
Summary 2: This section discusses the fundamentals of preparing high-quality Italian pasta dishes.

Chunk 3: Climate change is affecting ecosystems worldwide. Rising temperatures are causing glaciers to melt at unprecedented rates. Scientists warn that immediate action is needed to reduce carbon em

### Document Splitting

In [13]:
docs = [ Document(
    page_content= chunk.chunk_text,
    metadata= {"summary": "chunk.summary"}
) for chunk in chunks] 

docs

[Document(metadata={'summary': 'chunk.summary'}, page_content="Artificial intelligence is transforming technology and shaping the future. Machine learning algorithms are becoming more sophisticated every day. Deep learning models can now process vast amounts of data efficiently. Neural networks are inspired by the human brain's structule."),
 Document(metadata={'summary': 'chunk.summary'}, page_content='The best pasta recipes include fresh ingredients and proper cooking techniques. Italian cuisine emphasizes quality olive oil and regional cheeses. Authentic carbonara uses guanciale, eggs, pecorino romano, and black pepper. Cooking pasta al dente ensures the best texture and flavor.'),
 Document(metadata={'summary': 'chunk.summary'}, page_content='Climate change is affecting ecosystems worldwide. Rising temperatures are causing glaciers to melt at unprecedented rates. Scientists warn that immediate action is needed to reduce carbon emissions. Renewable energy sources offer hope for a su

In [14]:
doc= [ (f"{i+1}. page_content: {docs[i].page_content}", f"metadata: {docs[i].metadata}" ) for i in range(0,len(docs)) ]
doc
# print(*(d for d in doc), end="\n")

[("1. page_content: Artificial intelligence is transforming technology and shaping the future. Machine learning algorithms are becoming more sophisticated every day. Deep learning models can now process vast amounts of data efficiently. Neural networks are inspired by the human brain's structule.",
  "metadata: {'summary': 'chunk.summary'}"),
 ('2. page_content: The best pasta recipes include fresh ingredients and proper cooking techniques. Italian cuisine emphasizes quality olive oil and regional cheeses. Authentic carbonara uses guanciale, eggs, pecorino romano, and black pepper. Cooking pasta al dente ensures the best texture and flavor.',
  "metadata: {'summary': 'chunk.summary'}"),
 ('3. page_content: Climate change is affecting ecosystems worldwide. Rising temperatures are causing glaciers to melt at unprecedented rates. Scientists warn that immediate action is needed to reduce carbon emissions. Renewable energy sources offer hope for a sustainable future.',
  "metadata: {'summar